In [5]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.util import bigrams
from nltk.stem import PorterStemmer

import re
import datetime



[nltk_data] Downloading package stopwords to /Users/kilo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### <span style="color:#003049">1. Get data </span>
[data source](https://www.kaggle.com/clmentbisaillon/fake-and-real-news-dataset) 

In [6]:
df_0 = pd.read_csv("../data/Fake.csv")
df_1 = pd.read_csv("../data/True.csv")

### <span style="color:#003049">2. EDA</span> 

<img src="../images/Screenshot 2021-05-17 at 16.24.54.png
" width="300" height="50" />

In [7]:
df_0

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"
...,...,...,...,...
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"


In [8]:
df_1.tail()

,title,text,subject,date
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017"
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017"
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017"
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017"
21416,Indonesia to buy $1.14 billion worth of Russia...,JAKARTA (Reuters) - Indonesia will buy 11 Sukh...,worldnews,"August 22, 2017"


In [9]:
# Adding category 0 to fake news and category 1 to true news
df_0["category"] = 0
df_1["category"] = 1

In [10]:
# Concatenating dataframes
df = pd.concat([df_0, df_1],axis=0)
df = df.reset_index()
df = df.drop(['index'], axis=1)
df

,title,text,subject,date,category
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
...,...,...,...,...,...
44893,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017",1
44894,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017",1
44895,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017",1
44896,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017",1


In [11]:
# Saving dataframe as CSV
df.to_csv(f'../data/df_fakenews_merge.csv', index=False)

In [12]:
# Quick overview of the new dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     44898 non-null  object
 1   text      44898 non-null  object
 2   subject   44898 non-null  object
 3   date      44898 non-null  object
 4   category  44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [13]:
# Returning the number of missing values in each column
df.isnull( ).sum( )

title       0
text        0
subject     0
date        0
category    0
dtype: int64

In [14]:
# Number of unique elements in "subject" column
df["subject"].unique()

array(['News', 'politics', 'Government News', 'left-news', 'US_News',
       'Middle-east', 'politicsNews', 'worldnews'], dtype=object)

In [15]:
# Statistical summary for numerical columns present in the dataset. 
# This step does not make much sense on this dataframe
df.describe()

,category
count,44898.000000
mean,0.477015
std,0.499477
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,1.000000


In [16]:
# Getting number of dimensions as well as the size in each dimension
df.shape

(44898, 5)

### <span style="color:#003049">3. Data Cleaning</span> 

<img src="../images/data_cleaning.jpeg" width="300" height="50" />

In [18]:
# Removing rows containing NaN or missing values
df.dropna(inplace=True)

In [19]:
# Analyzing duplicated values
df.duplicated().sum()

np.int64(209)

In [20]:
# Dropping duplicates
df = df.drop_duplicates()

In [21]:
# Counting duplicated rows in "title" column
df["title"].duplicated().sum()

np.int64(5960)

In [22]:
df["text"].duplicated().sum()

np.int64(6043)

---------------

 ##### <span style="color:#003049">I would like to find out how many duplicated "titles" and "texts" are to be found in fake_news</span> 

In [ ]:
# Nota: este análisis es sobre df_0 (fake news original, antes del merge y deduplicación)
# para ver qué proporción de duplicados viene de noticias falsas
df_0["title"].duplicated().sum()

In [ ]:
df_0["text"].duplicated().sum()

###### <span style="color:#003049">Most of the duplicated rows are part of fake_news. Maybe this is due to the need to repeat a message to reaffirm it or probably as well to the little imagination of the inventors of lies. </span>


-------

In [25]:
# Dropping duplicated rows in "text" column
df = df.drop_duplicates(subset=['text'])

In [26]:
# Dropping duplicated rows in "title" column
df = df.drop_duplicates(subset=['title'])

In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 38270 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   title     38270 non-null  object
 1   text      38270 non-null  object
 2   subject   38270 non-null  object
 3   date      38270 non-null  object
 4   category  38270 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.8+ MB


### <span style="color:#003049">3. Preprocessing </span> 

<img src="../images/Python-data-preprocessing.png" width="300" height="50" />

#### <span style="color:#003049">Next steps: </span> 
<span style="color:#003049">
    <ul>
        <li><b>Removing stopwords before removing characters. I will add the word "us" to the list of stopwords so that after the removal of characters the only "us" that remains comes from the abbreviation U.S. (United States).</b></li>
        <li><b>Changing the $ symbol to the word "Dollar"</b></li>
        <li><b>Removing characters</b></li>
        <li><b>Removing digits</b></li>
        <li><b>Removing single letters</b></li>
        <li><b>Removing the word "reuter"</b></li>
        <li><b>Removing stopwords again, maintaining the word "us" this time </b></li>
        <li><b>Change date column to ordinal format</b></li>
        <li><b>Add columns length_title and length_text</b></li>       
    </ul>
</span> 

In [ ]:
ps = PorterStemmer()
en_stops = stopwords.words('english')
us_stops = stopwords.words('english')
us_stops.append("us")
us_stops.append("would")
pattern2 = "[_]"

def clean_text(news):
    # Primera pasada de stopwords: quita "us" como pronombre antes de limpiar caracteres
    # (así el único "us" que queda viene de la abreviatura U.S., que pierde los puntos)
    news = news.lower().split()
    news = [word for word in news if word not in us_stops]
    news = " ".join(news)
    news = re.sub(r'\$[^\s]+', 'dollar', news)        # $ → "dollar"
    news = re.sub(r'https?://\S+|www\.\S+', '', news)  # elimina URLs
    news = re.sub(r'[^\w\s]+', '', news)                # elimina caracteres especiales
    news = re.sub(r'\w*\d\w*', '', news)                # elimina palabras con dígitos
    news = re.sub(r' \d+', ' ', news)                   # elimina dígitos sueltos
    news = re.sub(r'(?:^| )\w(?:$| )', ' ', news)      # elimina letras sueltas
    news = re.sub(r'reuters', '', news)
    news = news.split()
    news = [re.sub(pattern2, '', word) for word in news]
    news = [ps.stem(word) for word in news]
    # Segunda pasada de stopwords: esta vez SIN "us" para preservar la abreviatura
    news = [word for word in news if word not in en_stops]
    return " ".join(news)

In [ ]:
df['text'] = df['text'].apply(clean_text)
df['title'] = df['title'].apply(clean_text)
df

In [ ]:
# Verificación post-limpieza: detectar textos vacíos o muy cortos que podrían
# haber quedado después de quitar stopwords, dígitos y caracteres especiales
empty_texts  = (df['text'].str.strip() == '').sum()
empty_titles = (df['title'].str.strip() == '').sum()
very_short   = (df['text'].str.split().str.len() < 5).sum()

print(f"Textos vacíos:                    {empty_texts}")
print(f"Títulos vacíos:                   {empty_titles}")
print(f"Textos muy cortos (<5 palabras):  {very_short}")

if empty_texts > 0 or empty_titles > 0:
    df = df[(df['text'].str.strip() != '') & (df['title'].str.strip() != '')]
    print(f"\nEliminadas filas con texto o título vacío. Filas restantes: {len(df)}")

#### <span style="color:#003049">Now I will work on the date column</span> 

In [ ]:
# Convertimos a datetime usando errors='coerce': las fechas inválidas se convierten en NaT
# Esto es más robusto que filtrar por longitud del string
df['date'] = pd.to_datetime(df['date'], errors='coerce')
rows_before = len(df)
df = df.dropna(subset=['date'])
print(f"Filas eliminadas por fecha inválida: {rows_before - len(df)}")
print(f"Rango de fechas: {df['date'].min()} → {df['date'].max()}")

In [ ]:
# La columna ya es datetime (convertida arriba). Solo convertimos a ordinal.
df['date'] = df['date'].apply(lambda x: x.toordinal())
print("dtype:", df['date'].dtype)
print("min:", df['date'].min(), " | max:", df['date'].max())

In [ ]:
df["length_text"] = [len(word.split()) for word in df["text"]]
df["length_title"] = [len(word.split()) for word in df["title"]]
df

In [ ]:
print(df.iloc[6,1])

### <span style="color:#003049">The data is ready to be processed.</span> 

In [ ]:
# Saving dataframe as CSV
df.to_csv(f'../data/df_fake_real_news.csv', index=False)

Link to [fake and real news viz](fake_and_real_news_viz.ipynb)<br>

Link to [machine learning notebook](fake_and_real_news_machine_learning.ipynb)